# KuchoLM training data generator

日本語コーパスの原文を MeCab で解析し、`原文 -> NIDA_FICTION` の教師ペアを生成する Colab 用ノートブックです。

現実の韓国語話者の日本語を模倣するものではなく、創作上の語尾スタイルとして扱います。


In [ ]:
!pip -q install mecab-python3 unidic-lite datasets

## 1. コーパスを読み込む

既定では Hugging Face `datasets` から日本語テキストを読み込める形にしています。ライセンスを確認した上で `DATASET_NAME` と `TEXT_COLUMN` を変更してください。
ローカルの `.txt` を使う場合は `USE_LOCAL_TEXT = True` にします。

In [ ]:
from pathlib import Path
import json
import re
import MeCab
from datasets import load_dataset

OUTPUT_PATH = Path('/content/kucholm_nida.jsonl')
MAX_ROWS = 100_000

USE_LOCAL_TEXT = False
LOCAL_TEXT_PATH = Path('/content/corpus.txt')

DATASET_NAME = 'range3/cc100-ja'
DATASET_SPLIT = 'train'
TEXT_COLUMN = 'text'

tagger = MeCab.Tagger()


In [ ]:
if USE_LOCAL_TEXT:
    corpus = (line.strip() for line in LOCAL_TEXT_PATH.open(encoding='utf-8'))
else:
    dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT, streaming=True)
    corpus = (str(row[TEXT_COLUMN]).strip() for row in dataset)


## 2. MeCab で文末を解析して変換

初版のルールは単純です。

- 疑問文: `...か？` / `...?` -> `...ニカ？`
- `です` -> `ニダ`
- `でした` -> `だったニダ`
- `ます` -> 動詞の基本形 + `ニダ`
- `ました` -> 元の語幹を保ちつつ `たニダ` を優先
- その他 -> 文末へ `ニダ`

意味を壊しにくくするため、引用符内や URL/コードっぽい行は除外します。

In [ ]:
URL_RE = re.compile(r'https?://|www\.|```|`[^`]+`')
END_PUNCT = {'。', '！', '!', '？', '?'}

def parse_tokens(text):
    node = tagger.parseToNode(text)
    tokens = []
    while node:
        if node.surface:
            features = node.feature.split(',')
            tokens.append({
                'surface': node.surface,
                'pos': features[0] if features else '',
                'base': features[6] if len(features) > 6 else '*',
            })
        node = node.next
    return tokens

def to_nida(text):
    text = text.strip()
    if not text or URL_RE.search(text):
        return None

    tokens = parse_tokens(text)
    if not tokens:
        return None

    punctuation = ''
    if tokens[-1]['surface'] in END_PUNCT:
        punctuation = tokens.pop()['surface']

    if not tokens:
        return None

    is_question = punctuation in {'？', '?'}
    if tokens and tokens[-1]['surface'] == 'か':
        tokens.pop()
        is_question = True

    surfaces = [token['surface'] for token in tokens]

    if is_question:
        return ''.join(surfaces) + 'ニカ' + (punctuation or '？')

    if surfaces[-2:] == ['でし', 'た']:
        return ''.join(surfaces[:-2]) + 'だったニダ' + punctuation

    if surfaces[-1:] == ['です']:
        return ''.join(surfaces[:-1]) + 'ニダ' + punctuation

    if surfaces[-2:] == ['まし', 'た']:
        for index in range(len(tokens) - 3, -1, -1):
            token = tokens[index]
            if token['pos'] == '動詞' and token['base'] not in {'*', ''}:
                prefix = ''.join(item['surface'] for item in tokens[:index])
                return prefix + token['base'] + 'たニダ' + punctuation

    if surfaces[-1:] == ['ます']:
        for index in range(len(tokens) - 2, -1, -1):
            token = tokens[index]
            if token['pos'] == '動詞' and token['base'] not in {'*', ''}:
                prefix = ''.join(item['surface'] for item in tokens[:index])
                return prefix + token['base'] + 'ニダ' + punctuation

    return ''.join(surfaces) + 'ニダ' + punctuation


## 3. 変換例

In [ ]:
examples = [
    '今日は学校です。',
    '明日は学校に行きます。',
    'これは本当ですか？',
    '昨日は雨でした。',
]

for source in examples:
    print(source, '->', to_nida(source))


## 4. `原文 -> ニダ口調` の JSONL を生成

In [ ]:
written = 0
with OUTPUT_PATH.open('w', encoding='utf-8') as output:
    for source in corpus:
        source = source.strip()
        if not source or len(source) < 2 or len(source) > 256:
            continue

        target = to_nida(source)
        if not target or target == source:
            continue

        output.write(json.dumps({
            'style': 'NIDA_FICTION',
            'source': source,
            'target': target,
        }, ensure_ascii=False) + '\n')

        written += 1
        if written >= MAX_ROWS:
            break

print('written:', written)
print('output:', OUTPUT_PATH)


## 5. 出力確認

In [ ]:
with OUTPUT_PATH.open(encoding='utf-8') as f:
    for _ in range(10):
        line = f.readline()
        if not line:
            break
        print(line.rstrip())
